### Preparação e Organização dos Dados

Como o objetivo é encontrar agrupamentos baseados em características acústicas e numéricas, precisamos remover colunas de identificação e texto (como `track_id`, `artists`, `album_name`, `track_name` e as colunas `Unnamed`). Também removeremos a coluna `track_genre` para garantir que o aprendizado seja estritamente **não supervisionado**. Guardaremos essa coluna em uma variável separada para consultas futuras.

Além disso, as variáveis contínuas possuem escalas muito diferentes (ex: `tempo` varia de 0 a 200+, enquanto `acousticness` varia de 0 a 1). Utilizaremos o `StandardScaler` para normalizar essas variáveis, garantindo que algoritmos baseados em distância (como K-Means e DBSCAN) funcionem corretamente. Também filtraremos possíveis anomalias, como músicas com duração de 0 segundos.

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. Copiar o dataframe para não alterar o original carregado
df_clustering = df.copy()

# 2. Filtrar anomalias (ex: músicas com duração zero)
df_clustering = df_clustering[df_clustering['duration_ms'] > 0]

# 3. Separar as labels originais (gêneros) para análise externa futura
generos_originais = df_clustering['track_genre']

# 4. Remover colunas não numéricas e identificadores
colunas_para_remover = ['Unnamed: 0.1', 'Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'track_genre']
df_features = df_clustering.drop(columns=colunas_para_remover)

# Remover valores nulos, se houver
df_features = df_features.dropna()
generos_originais = generos_originais.loc[df_features.index]

# 5. Escalonamento dos dados
scaler = StandardScaler()
features_scaled = scaler.fit_transform(df_features)

# Transformar de volta em DataFrame para facilitar visualização
df_scaled = pd.DataFrame(features_scaled, columns=df_features.columns)

print(f"Formato dos dados após limpeza e escalonamento: {df_scaled.shape}")
display(df_scaled.head())

### Redução de Dimensionalidade: PCA

Nosso dataset possui muitas dimensões (variáveis acústicas). Para visualizar os agrupamentos e reduzir a complexidade computacional, aplicaremos a Análise de Componentes Principais (PCA). O objetivo é encontrar os componentes que explicam a maior parte da variância dos dados musicais. Vamos projetar os dados em 2 componentes principais para facilitar a plotagem em gráficos de dispersão 2D.

In [ ]:
from sklearn.decomposition import PCA

# Aplicando PCA para reduzir a 2 componentes principais
pca = PCA(n_components=2)
features_pca = pca.fit_transform(df_scaled)

# Criando um DataFrame com as componentes principais
df_pca = pd.DataFrame(data=features_pca, columns=['Componente_1', 'Componente_2'])

# Calculando a variância explicada
variancia_explicada = pca.explained_variance_ratio_
print(f"Variância explicada pelo Componente 1: {variancia_explicada[0]*100:.2f}%")
print(f"Variância explicada pelo Componente 2: {variancia_explicada[1]*100:.2f}%")
print(f"Variância total explicada: {sum(variancia_explicada)*100:.2f}%")

# Visualização inicial do espaço latente gerado pelo PCA
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Componente_1', y='Componente_2', data=df_pca, alpha=0.5, s=10)
plt.title('Projeção PCA dos Dados Acústicos (2D)')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.show()

### Agrupamento com K-Means e Avaliação

Para encontrar os agrupamentos naturais de músicas, utilizaremos o algoritmo **K-Means**. Como não sabemos a priori quantos perfis musicais existem nos dados, aplicaremos o **Método do Cotovelo (Elbow Method)** e avaliaremos a coesão usando o **Silhouette Score** para um pequeno subconjunto (devido ao alto custo computacional do Silhouette em datasets gigantes).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore') # Ignorar avisos de memory leak do KMeans no Windows

inercia = []
silhouette_scores = []
K_range = range(2, 8)

# Como o dataset é muito grande (114k linhas), vamos pegar uma amostra aleatória
# para calcular o Silhouette Score em tempo hábil durante os testes
amostra_idx = np.random.choice(df_pca.shape[0], 10000, replace=False)
pca_amostra = df_pca.iloc[amostra_idx]

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df_pca)
    inercia.append(kmeans.inertia_)
    
    # Avaliação Quantitativa: Silhouette na amostra
    labels_amostra = kmeans.predict(pca_amostra)
    score = silhouette_score(pca_amostra, labels_amostra)
    silhouette_scores.append(score)

# Plotando Elbow e Silhouette
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Gráfico 1: Inércia (Elbow)
ax1.plot(K_range, inercia, marker='o', color='b')
ax1.set_title('Método do Cotovelo (Inércia)')
ax1.set_xlabel('Número de Clusters (k)')
ax1.set_ylabel('Inércia')

# Gráfico 2: Silhouette Score
ax2.plot(K_range, silhouette_scores, marker='s', color='g')
ax2.set_title('Avaliação: Silhouette Score')
ax2.set_xlabel('Número de Clusters (k)')
ax2.set_ylabel('Score')

plt.show()

### Visualização dos Padrões e Análise Crítica

Com base nas métricas acima, escolheremos o valor ideal de $k$ (neste exemplo, vamos assumir $k=4$, mas pode ser ajustado com base nos gráficos plotados na célula anterior). Aplicaremos o K-Means final, visualizaremos as músicas agrupadas no espaço do PCA e, o mais importante, usaremos **Boxplots** para interpretar o que cada cluster significa no mundo real.

In [ ]:
# Executando com o K escolhido (Altere o valor de k_ideal se necessário)
k_ideal = 4 
kmeans_final = KMeans(n_clusters=k_ideal, random_state=42, n_init=10)
df_pca['Cluster'] = kmeans_final.fit_predict(df_pca)

# Trazendo os clusters de volta para o DataFrame original (sem escalonamento) para interpretação
df_features['Cluster'] = df_pca['Cluster'].values

# Visualização dos clusters no espaço PCA
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Componente_1', y='Componente_2', hue='Cluster', palette='viridis', data=df_pca, alpha=0.6, s=15)
plt.title(f'Agrupamento K-Means com k={k_ideal} no Espaço PCA')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.legend(title='Cluster')
plt.show()

# Análise Crítica (Qualitativa): O que são esses clusters?
# Selecionamos features acústicas interessantes para comparar as distribuições
features_para_analisar = ['energy', 'danceability', 'acousticness', 'valence']

plt.figure(figsize=(16, 10))
for i, feature in enumerate(features_para_analisar, 1):
    plt.subplot(2, 2, i)
    sns.boxplot(x='Cluster', y=feature, data=df_features, palette='viridis')
    plt.title(f'Distribuição de {feature.capitalize()} por Cluster')

plt.tight_layout()
plt.show()

# Exibindo os valores médios de cada cluster
print("\nMédia das características por Cluster:")
display(df_features.groupby('Cluster')[features_para_analisar].mean())